In [6]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openrouter import ChatOpenRouter

env_path = next(
    (path / ".env" for path in (Path.cwd(), *Path.cwd().parents) if (path / ".env").exists()),
    Path.cwd() / ".env",
)
load_dotenv(env_path, override=True)

if os.environ.get("OPENROUTER_API_KEY"):
    print("OpenRouter API key loaded")
else:
    raise ValueError("OPENROUTER_API_KEY not found")

llm_openrouter = ChatOpenRouter(model="openai/gpt-4o-mini", temperature=0)

OpenRouter API key loaded


In [7]:
from langchain_core.prompts import PromptTemplate

result = llm_openrouter.invoke("Tell me a joke. Generate the output in key-value pair format with the following keys: setup, punchline")
result.content

'{\n  "setup": "Why did the scarecrow win an award?",\n  "punchline": "Because he was outstanding in his field!"\n}'

In [8]:
from pydantic import BaseModel, Field

class llm_schema(BaseModel):
    setup: str = Field(description="The setup for the joke")
    punchline: str = Field(description="The punchline for the joke")

In [9]:
llm_structured_output = llm_openrouter.with_structured_output(llm_schema)

llm_structured_output.invoke("Tell me a joke")

llm_schema(setup='Why did the scarecrow win an award?', punchline='Because he was outstanding in his field!')

In [10]:
from typing import TypedDict 

class llm_schema_td(TypedDict):
    setup: str
    punchline: str

In [11]:
llm_structured_typed_dict = llm_openrouter.with_structured_output(llm_schema_td)

result = llm_structured_typed_dict.invoke("Tell me a joke")
result

{'setup': "Why don't scientists trust atoms?",
 'punchline': 'Because they make up everything!'}

In [12]:
from pydantic import BaseModel, ValidationError
from typing import TypedDict

class PersonTD(TypedDict):
    name: str
    age: int

class PersonPydantic(BaseModel):
    name: str
    age: int

# age is the string "30"
data = {"name": "Alice", "age": "30"}

# TypedDict: leaves it as a string, no checking
td = PersonTD(**data)
print("TD  age:", td["age"], type(td["age"]))     # 30 <class 'str'>  -> untouched

# Pydantic v2: coerces "30" -> 30
pd = PersonPydantic(**data)
print("Pyd age:", pd.age, type(pd.age))           # 30 <class 'int'>  -> coerced

TD  age: 30 <class 'str'>
Pyd age: 30 <class 'int'>


In [13]:
bad = {"name": "Alice", "age": "not a number"}

# Pydantic: hard fail
try:
    PersonPydantic(**bad)
except ValidationError as e:
    print("Pydantic rejected it:", e.errors()[0]["msg"])

# TypedDict: accepts anything, no error
td_bad = PersonTD(**bad)
print("TD accepted garbage:", td_bad["age"])      # 'not a number'

Pydantic rejected it: Input should be a valid integer, unable to parse string as an integer
TD accepted garbage: not a number
